# 🌲 Wildfire Risk Dashboard: Data Preparation Pipeline
สมุดเล่มนี้ใช้สำหรับแปลงข้อมูลจากไฟล์ดิบ (`df_final_model.csv`) ให้กลายเป็นฐานข้อมูล JSON สำหรับนำไปใช้งานบนหน้าเว็บ Dashboard

In [9]:
import pandas as pd
import json
import os

# 1. กำหนดที่อยู่ไฟล์
INPUT_CSV = '../Dataset/df_final_model.csv'
OUTPUT_JSON = '../../wildfire risk visualization/data/district_points_data.json'

print("กำลังโหลดข้อมูล...")
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"โหลดสำเร็จ: {len(df)} แถว")

กำลังโหลดข้อมูล...
โหลดสำเร็จ: 68447 แถว


## 2. กระบวนการสกัดข้อมูลรายพิกัด (Point-based Extraction)
เราจะเลือกข้อมูลเดือนล่าสุดของแต่ละอำเภอ และสกัดพิกัดจริงออกมาเพื่อแสดงผลเป็นจุด Cluster บนแผนที่

In [10]:
import pandas as pd
import numpy as np

# 1. เลือกข้อมูลเดือนล่าสุด
latest_month = df['month'].max()
df_latest = df[df['month'] == latest_month].copy()

# (สมมติว่าถ้าคุณนำโมเดลมาทำนายแล้ว จะได้คอลัมน์ pred_risk_prob)
# df_latest['pred_risk_prob'] = model.predict_proba(X_scaled)[:, 1] * 100

# ถ้ายังไม่มีโมเดล ใช้สูตรจำลอง Baseline Risk เดิมของคุณไปก่อน (คูณ 100 ให้เป็นเปอร์เซ็นต์)
df_latest['pred_risk_prob'] = ((df_latest['temp'] / 40) + ((1 - df_latest['soil_moisture']) * 0.5)) * 100

db_points = {}

# กำหนดจำนวนจุดสูงสุดที่จะแสดงต่อ 1 อำเภอ (ปรับเพิ่มลดได้ตามความแรงของเว็บ)
max_points_per_district = 50 

print("กำลังจัดเตรียมข้อมูลรายจุดสำหรับเว็บ...")

# 2. ใช้ Groupby เพื่อจัดการทีละอำเภอ
for (prov, dist), group in df_latest.groupby(['NAME_1', 'NAME_2']):
    
    if prov not in db_points: 
        db_points[prov] = {}
        
    db_points[prov][dist] = []
    
    # 3. สุ่มจุดตัวแทน (Sampling) เพื่อให้จุดกระจายตัวทั่วอำเภอ
    if len(group) > max_points_per_district:
        # เลือก 50 จุดที่มีค่าความเสี่ยงสูงที่สุดในอำเภอนั้น
        sampled_group = group.nlargest(max_points_per_district, 'pred_risk_prob')
    else:
        sampled_group = group
    
    # 4. สกัดข้อมูลจุดที่สุ่มมาได้ใส่ JSON
    for _, row in sampled_group.iterrows():
        db_points[prov][dist].append({
            'lat': float(row['LATITUDE']),
            'lng': float(row['LONGITUDE']),
            'risk_prob': float(row['pred_risk_prob']), 
            'temp': float(row['temp']),
            'ndvi': float(row['ndvi']),
            # ตรวจสอบว่ามีคอลัมน์ CONFIDENCE หรือไม่ก่อนเรียกใช้
            'confidence': int(row['CONFIDENCE']) if 'CONFIDENCE' in row and pd.notnull(row['CONFIDENCE']) else 0
        })

print(f"ประมวลผลเสร็จสิ้น: รวม {len(db_points)} จังหวัด")

กำลังจัดเตรียมข้อมูลรายจุดสำหรับเว็บ...
ประมวลผลเสร็จสิ้น: รวม 19 จังหวัด


## 3. บันทึกผลลัพธ์เพื่อนำไปใช้บนเว็บ

In [11]:
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(db_points, f, ensure_ascii=False, indent=2)

print(f"✅ บันทึกไฟล์ไปที่: {OUTPUT_JSON}")
print("ตอนนี้คุณสามารถรันหน้าเว็บและกด Predict เพื่อดูจุดเหล่านี้ได้ทันที!")

✅ บันทึกไฟล์ไปที่: ../../wildfire risk visualization/data/district_points_data.json
ตอนนี้คุณสามารถรันหน้าเว็บและกด Predict เพื่อดูจุดเหล่านี้ได้ทันที!
